In [1]:
import numpy as np
import pandas as pd

In [2]:
train_data = pd.read_csv("../data/processed/train.csv")
test_data = pd.read_csv("../data/processed/test.csv")

In [3]:
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (800167, 4)
Test shape: (200042, 4)


In [6]:
# 1. Kiểm tra số lượng user và movie duy nhất
print("=== SỐ LƯỢNG ID DUY NHẤT ===")
print("Train - users:", train_data["user_id"].nunique())
print("Train - movies:", train_data["movie_id"].nunique())
print("Test - users:", test_data["user_id"].nunique())
print("Test - movies:", test_data["movie_id"].nunique())

# 2. Kiểm tra các cặp user-movie bị trùng
print("\n=== KIỂM TRA TRÙNG LẶP ===")
print(
    "Train - số dòng trùng hoàn toàn:",
    train_data.duplicated().sum()
)
print(
    "Test - số dòng trùng hoàn toàn:",
    test_data.duplicated().sum()
)
print(
    "Train - số cặp user-movie trùng:",
    train_data.duplicated(["user_id", "movie_id"]).sum()
)
print(
    "Test - số cặp user-movie trùng:",
    test_data.duplicated(["user_id", "movie_id"]).sum()
)

# 3. Kiểm tra user và movie trong test có xuất hiện ở train không
train_users = set(train_data["user_id"].unique())
train_movies = set(train_data["movie_id"].unique())

test_users = set(test_data["user_id"].unique())
test_movies = set(test_data["movie_id"].unique())

print("\n=== ID TRONG TEST KHÔNG CÓ TRONG TRAIN ===")
print("User chưa xuất hiện trong train:", len(test_users - train_users))
print("Movie chưa xuất hiện trong train:", len(test_movies - train_movies))

# 4. Kiểm tra cùng một cặp user-movie có xuất hiện ở cả hai tập không
train_pairs = set(
    zip(train_data["user_id"], train_data["movie_id"])
)
test_pairs = set(
    zip(test_data["user_id"], test_data["movie_id"])
)

print(
    "Cặp user-movie xuất hiện ở cả train và test:",
    len(train_pairs & test_pairs)
)

=== SỐ LƯỢNG ID DUY NHẤT ===
Train - users: 6040
Train - movies: 3683
Test - users: 6038
Test - movies: 3444

=== KIỂM TRA TRÙNG LẶP ===
Train - số dòng trùng hoàn toàn: 0
Test - số dòng trùng hoàn toàn: 0
Train - số cặp user-movie trùng: 0
Test - số cặp user-movie trùng: 0

=== ID TRONG TEST KHÔNG CÓ TRONG TRAIN ===
User chưa xuất hiện trong train: 0
Movie chưa xuất hiện trong train: 23
Cặp user-movie xuất hiện ở cả train và test: 0


In [7]:
# Tạo ánh xạ ID dựa trên tập train
user_ids = train_data["user_id"].unique()
movie_ids = train_data["movie_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

# Ánh xạ ngược để khôi phục ID gốc
idx_to_user = {
    idx: user_id
    for user_id, idx in user_to_idx.items()
}

idx_to_movie = {
    idx: movie_id
    for movie_id, idx in movie_to_idx.items()
}

n_users = len(user_to_idx)
n_movies = len(movie_to_idx)

print("Số user:", n_users)
print("Số movie:", n_movies)
print("Kích thước ma trận tương tác:", (n_users, n_movies))
print("Tổng số lượt đánh giá train:", len(train_data))

Số user: 6040
Số movie: 3683
Kích thước ma trận tương tác: (6040, 3683)
Tổng số lượt đánh giá train: 800167


In [8]:
# Tổng số phần tử trong ma trận user-movie
total_cells = n_users * n_movies

# Số lượt đánh giá quan sát được
observed_ratings = len(train_data)

# Tỷ lệ ô đã có đánh giá và tỷ lệ ô trống
density = observed_ratings / total_cells
sparsity = 1 - density

print(f"Tổng số ô trong ma trận: {total_cells:,}")
print(f"Số ô có đánh giá: {observed_ratings:,}")
print(f"Độ phủ (density): {density:.4%}")
print(f"Độ thưa (sparsity): {sparsity:.4%}")

Tổng số ô trong ma trận: 22,245,320
Số ô có đánh giá: 800,167
Độ phủ (density): 3.5970%
Độ thưa (sparsity): 96.4030%


In [9]:
# Ánh xạ ID gốc sang chỉ số nội bộ
train_users_idx = train_data["user_id"].map(user_to_idx).to_numpy()
train_movies_idx = train_data["movie_id"].map(movie_to_idx).to_numpy()

# Rating mục tiêu
train_ratings = train_data["rating"].to_numpy(dtype=np.float64)

# Kiểm tra dữ liệu huấn luyện
print("Kích thước train_users_idx:", train_users_idx.shape)
print("Kích thước train_movies_idx:", train_movies_idx.shape)
print("Kích thước train_ratings:", train_ratings.shape)

print("\nMột vài mẫu sau khi ánh xạ:")
print("User indices:", train_users_idx[:10])
print("Movie indices:", train_movies_idx[:10])
print("Ratings:", train_ratings[:10])

# Kiểm tra tính hợp lệ của chỉ số
print("\nUser index hợp lệ:",
      train_users_idx.min() >= 0 and train_users_idx.max() < n_users)
print("Movie index hợp lệ:",
      train_movies_idx.min() >= 0 and train_movies_idx.max() < n_movies)

Kích thước train_users_idx: (800167,)
Kích thước train_movies_idx: (800167,)
Kích thước train_ratings: (800167,)

Một vài mẫu sau khi ánh xạ:
User indices: [0 1 2 3 4 5 6 7 8 9]
Movie indices: [0 1 2 3 4 5 6 7 8 9]
Ratings: [2. 4. 3. 4. 4. 3. 3. 3. 3. 4.]

User index hợp lệ: True
Movie index hợp lệ: True


In [10]:
# Tham số khởi tạo ban đầu
n_factors = 50
learning_rate = 0.005
reg = 0.02
n_epochs = 15

# Rating trung bình của tập train
global_mean = train_ratings.mean()

# Khởi tạo ngẫu nhiên các vector nhân tố
rng = np.random.default_rng(42)

P = rng.normal(
    0, 0.1, size=(n_users, n_factors)
)
Q = rng.normal(
    0, 0.1, size=(n_movies, n_factors)
)

# Khởi tạo bias
user_bias = np.zeros(n_users)
movie_bias = np.zeros(n_movies)

print("Số nhân tố:", n_factors)
print("Learning rate:", learning_rate)
print("Regularization:", reg)
print("Số epochs:", n_epochs)
print("Global mean rating:", round(global_mean, 4))
print("Kích thước P:", P.shape)
print("Kích thước Q:", Q.shape)
print("Kích thước user_bias:", user_bias.shape)
print("Kích thước movie_bias:", movie_bias.shape)

Số nhân tố: 50
Learning rate: 0.005
Regularization: 0.02
Số epochs: 15
Global mean rating: 3.5813
Kích thước P: (6040, 50)
Kích thước Q: (3683, 50)
Kích thước user_bias: (6040,)
Kích thước movie_bias: (3683,)


In [11]:
# Huấn luyện Bias Matrix Factorization bằng SGD

rng = np.random.default_rng(42)
train_rmse_history = []

for epoch in range(n_epochs):
    # Xáo trộn thứ tự các rating trong mỗi epoch
    order = rng.permutation(len(train_ratings))

    for idx in order:
        u = train_users_idx[idx]
        i = train_movies_idx[idx]
        r = train_ratings[idx]

        # Dự đoán rating
        pred = (
            global_mean
            + user_bias[u]
            + movie_bias[i]
            + np.dot(P[u], Q[i])
        )

        # Sai số dự đoán
        error = r - pred

        # Lưu vector cũ để cập nhật đồng thời
        p_old = P[u].copy()
        q_old = Q[i].copy()

        # Cập nhật bias
        user_bias[u] += learning_rate * (
            error - reg * user_bias[u]
        )
        movie_bias[i] += learning_rate * (
            error - reg * movie_bias[i]
        )

        # Cập nhật latent factors
        P[u] += learning_rate * (
            error * q_old - reg * p_old
        )
        Q[i] += learning_rate * (
            error * p_old - reg * q_old
        )

    # Tính RMSE trên tập train sau mỗi epoch
    predictions = (
        global_mean
        + user_bias[train_users_idx]
        + movie_bias[train_movies_idx]
        + np.sum(
            P[train_users_idx] * Q[train_movies_idx],
            axis=1
        )
    )

    rmse = np.sqrt(np.mean((train_ratings - predictions) ** 2))
    train_rmse_history.append(rmse)

    print(f"Epoch {epoch + 1:02d}/{n_epochs} - Train RMSE: {rmse:.4f}")

Epoch 01/15 - Train RMSE: 0.9311
Epoch 02/15 - Train RMSE: 0.9089
Epoch 03/15 - Train RMSE: 0.8985
Epoch 04/15 - Train RMSE: 0.8916
Epoch 05/15 - Train RMSE: 0.8859
Epoch 06/15 - Train RMSE: 0.8800
Epoch 07/15 - Train RMSE: 0.8735
Epoch 08/15 - Train RMSE: 0.8655
Epoch 09/15 - Train RMSE: 0.8563
Epoch 10/15 - Train RMSE: 0.8462
Epoch 11/15 - Train RMSE: 0.8358
Epoch 12/15 - Train RMSE: 0.8250
Epoch 13/15 - Train RMSE: 0.8140
Epoch 14/15 - Train RMSE: 0.8030
Epoch 15/15 - Train RMSE: 0.7921
